### Setup

In [1]:
import numpy as np
import choix
from scipy.optimize import minimize
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from matplotlib import colors
import pandas as pd
import seaborn as sns
import pickle
import os
import sys

sys.path.append(os.path.abspath('../../'))
from metrics import compute_acc, compute_weighted_acc
from opt_fair import *
from distribution_utils import safe_kendalltau, to_numpy

In [2]:
!nvidia-smi

Sun Apr 26 14:38:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:17:00.0 Off |                    0 |
| N/A   77C    P0            204W /  300W |   16245MiB /  81920MiB |     97%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(device)

cuda


In [4]:
print(f"Current PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

Current PyTorch version: 2.4.0a0+f70bd71a48.nv24.06
CUDA available: True
CUDA version: 12.5


In [5]:
with open("data/FaceAgePC.pickle", 'rb') as handle:
    PC_faceage = pickle.load(handle)    
with open("data/FaceAgeDF.pickle", 'rb') as handle:
    df_faceage = pickle.load(handle)

In [6]:
df_faceage

,full_path,score,gender
0,nm1442940_rm3965098752_1996-10-3_2006.jpg,10,0.0
1,nm4832920_rm1781768448_2003-8-28_2013.jpg,10,0.0
2,nm0652089_rm860657920_1992-3-10_2002.jpg,10,0.0
3,nm0004917_rm1493730304_1969-5-12_1979.jpg,10,0.0
4,nm1113550_rm1332711936_1996-4-14_2006.jpg,10,0.0
...,...,...,...
9145,475367_1941-08-03_2011.jpg,70,1.0
9146,304085_1919-07-07_1989.jpg,70,1.0
9147,nm0001627_rm4164078592_1927-2-20_1997.jpg,70,1.0
9148,nm0000024_rm1715129344_1904-4-14_1974.jpg,70,1.0


In [7]:
import opt_fair
all_pc_faceage  = opt_fair._pc_without_reviewers(PC_faceage)

size = len(df_faceage)
print(size)

9150


In [8]:
print(len(all_pc_faceage))

250249


In [9]:
max_iter=1000

In [11]:
# PC_faceage

In [15]:
from collections import defaultdict

# Example input:
# data = {
#     0: {(1, 2), (2, 3), (3, 4)},
#     1: {(1, 2), (2, 3), (3, 1)},   # cycle
#     2: {(5, 6), (7, 8)}            # acyclic
# }

def is_acyclic_undirected(edges):
    """
    Check if an undirected graph defined by edges is acyclic.
    Returns True if acyclic, False if cycle exists.
    """
    graph = defaultdict(list)

    # Build undirected adjacency list
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = set()

    def dfs(node, parent):
        visited.add(node)

        for neighbor in graph[node]:
            if neighbor not in visited:
                if not dfs(neighbor, node):
                    return False
            elif neighbor != parent:
                # visited neighbor not parent => cycle
                return False

        return True

    for node in graph:
        if node not in visited:
            if not dfs(node, -1):
                return False

    return True


def check_workers(data):
    """
    For each worker, check whether their comparison graph is acyclic.
    Returns dict: worker_id -> True/False
    """
    result = {}

    for worker_id, comparisons in data.items():
        result[worker_id] = is_acyclic_undirected(comparisons)

    return result


# Example usage
if __name__ == "__main__":
    data = {
        0: {(1, 2), (2, 3), (3, 4)},       # tree -> acyclic
        1: {(1, 2), (2, 3), (3, 1)},       # triangle -> cyclic
        2: {(5, 6), (7, 8)},               # disconnected forest -> acyclic
        3: {(1, 2), (2, 3), (3, 4), (4, 2)} # cycle
    }

    results = check_workers(PC_faceage)
    
    cnt = 0
    for worker, acyclic in results.items():
        if acyclic:
            cnt += 1
#             print(f"Worker {worker}: {'Acyclic' if acyclic else 'Cyclic'}")
            
print("Done")
print(cnt == len(results.keys()))

Done
True
